# Silver — ecommerce_rastreamento_entregas

Este notebook lê a camada Bronze de rastreamento, aplica as 10 regras de qualidade/negócio, salva a tabela Silver em Delta e registra o resumo das falhas em `squad1.dq_monitoring_logs`.

In [0]:

# MAGIC %run ../../utils/utils

In [0]:
import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from functools import reduce
from datetime import datetime, timezone

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_rastreamento"
TABELA_DQ = "dq_monitoring_logs"

print(f"Iniciando processamento Silver - Rastreamento - Run ID: {RUN_ID}")

##  Anti-Join e Tabelas de Referência


In [0]:
# 1. Carrega a tabela Bronze de Rastreamento
try:
    df_bronze_rastreamento = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada. Rode a Bronze primeiro!")

# 2. Isola o Micro-lote (Considerando Silver E Quarentena)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    
    # Busca na subpasta oficial de quarentena
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("id_rastreamento") \
            .union(df_quarentena.select("id_rastreamento"))
    else:
        df_processados = df_silver_atual.select("id_rastreamento")
        
    df_micro_lote = df_bronze_rastreamento.join(df_processados, "id_rastreamento", "left_anti")
else:
    df_micro_lote = df_bronze_rastreamento

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote para processar: {qtd_novos}")

# =================================================================================
# 3. Leitura da Tabela de Referência Cruzada (PEDIDOS DA SILVER)
# =================================================================================
if delta_existe("silver", "ecommerce_pedidos", STORAGE_OPTIONS):
    # Forçamos a seleção e o dropDuplicates usando chaves explícitas
    df_pedidos_ref = obter_referencia_silver_ou_bronze("ecommerce_pedidos", ["id_pedido", "status_pedido"]) \
    .withColumnRenamed("id_pedido", "id_pedido_ref") \
    .withColumnRenamed("status_pedido", "status_pedido_ref")
 
    print("Tabela de referência df_pedidos_ref carregada com sucesso (Silver com fallback para Bronze).")
else:
    schema_pedidos = StructType([
        StructField("id_pedido_ref", LongType(), True),
        StructField("status_pedido_ref", StringType(), True)
    ])
    df_pedidos_ref = spark.createDataFrame([], schema_pedidos)

print("Tabela de referência df_pedidos_ref carregada com sucesso da camada Silver.")

## Aplicação das 10 Regras de Qualidade

In [0]:
if qtd_novos > 0:
    from functools import reduce

    # Domínio de status logísticos permitidos (Regra 2)
    status_logísticos_validos = ["em separacao", "coletado", "em transito", "saiu para entrega", "entregue"]

    # 1. ETAPA DE ACOPLAMENTO
    df_acoplado = df_micro_lote \
        .withColumn("dt_evento_ts", F.col("dt_evento").cast("timestamp")) \
        .withColumn("status_entrega", F.lower(F.trim(F.col("status_entrega")))) \
        .join(df_pedidos_ref, F.col("id_pedido_ecommerce") == F.col("id_pedido_ref"), "left_outer")

    # CORREÇÃO R6: a ordem cronológica do fluxo precisa ser avaliada pela
    # POSIÇÃO do status no fluxo de negócio (em separacao -> coletado ->
    # em transito -> saiu para entrega -> entregue), não pela própria
    # dt_evento_ts. Antes, a window era ordenada por dt_evento_ts e depois
    # comparava dt_evento_ts contra o "anterior" NESSA MESMA ordenação — o
    # que é uma tautologia sempre falsa (o anterior, por definição da
    # ordenação, nunca pode ser maior). Agora mapeamos cada status para uma
    # posição fixa no fluxo e ordenamos por ela.
    status_flow_expr = (
        F.when(F.col("status_entrega") == "em separacao", F.lit(0))
         .when(F.col("status_entrega") == "coletado", F.lit(1))
         .when(F.col("status_entrega") == "em transito", F.lit(2))
         .when(F.col("status_entrega") == "saiu para entrega", F.lit(3))
         .when(F.col("status_entrega") == "entregue", F.lit(4))
         .otherwise(F.lit(None))
    )
    df_acoplado = df_acoplado.withColumn("status_flow_idx", status_flow_expr)

    # Janelas analíticas
    w_id_rastreio = Window.partitionBy("id_rastreamento").orderBy(F.col("bronze_ingested_at").asc())
    w_cronologia_pedido = Window.partitionBy("id_pedido_ecommerce").orderBy(
        F.col("status_flow_idx").asc_nulls_last(), F.col("dt_evento_ts").asc()
    )

    # CORREÇÃO R7: precisa saber, por PEDIDO, se existe ALGUM evento com
    # status "entregue" — não comparar cada linha individualmente contra
    # "entregue" (isso reprovava todo o histórico de rastreamento de
    # pedidos entregues corretamente, exceto a última linha).
    df_flag_entregue_pedido = df_acoplado.groupBy("id_pedido_ecommerce").agg(
        F.max(F.when(F.col("status_entrega") == "entregue", 1).otherwise(0)).alias("tem_evento_entregue")
    )

    # CORREÇÃO R8: precisa medir especificamente entre o evento 'coletado' e
    # o evento 'entregue' do mesmo pedido — não entre "este evento" e "o
    # evento anterior genérico" (que pode ser qualquer status intermediário).
    df_dt_coletado = df_acoplado.filter(F.col("status_entrega") == "coletado") \
        .groupBy("id_pedido_ecommerce").agg(F.min("dt_evento_ts").alias("dt_coletado"))
    df_dt_entregue = df_acoplado.filter(F.col("status_entrega") == "entregue") \
        .groupBy("id_pedido_ecommerce").agg(F.min("dt_evento_ts").alias("dt_entregue"))
    df_sla_coleta_entrega = df_dt_coletado.join(df_dt_entregue, "id_pedido_ecommerce", "inner") \
        .withColumn("dias_coletado_entregue", F.datediff(F.col("dt_entregue"), F.col("dt_coletado")))

    # CORREÇÃO R10: soma o histórico já validado na Silver ao que está
    # chegando no lote, para que a contagem de transportadoras por pedido
    # considere o acumulado — não só o micro-lote atual (mesma classe de
    # correção já aplicada em R6/R8 de endereços).
    if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
        df_transportadora_hist = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS) \
            .select("id_pedido_ecommerce", "id_transportadora")
    else:
        schema_transp_hist = StructType([
            StructField("id_pedido_ecommerce", LongType(), True),
            StructField("id_transportadora", StringType(), True)
        ])
        df_transportadora_hist = spark.createDataFrame([], schema_transp_hist)

    df_transportadoras_combinado = (
        df_acoplado.select("id_pedido_ecommerce", "id_transportadora")
        .unionByName(df_transportadora_hist)
        .groupBy("id_pedido_ecommerce")
        .agg(F.size(F.collect_set("id_transportadora")).alias("qtd_transportadoras_total"))
    )

    # 2. ETAPA ANALÍTICA: Computa as transformações de janela e junta os agregados
    df_base = df_acoplado \
        .withColumn("qtd_id_rastreio", F.row_number().over(w_id_rastreio)) \
        .withColumn("dt_evento_anterior", F.lag("dt_evento_ts").over(w_cronologia_pedido)) \
        .join(df_flag_entregue_pedido, "id_pedido_ecommerce", "left") \
        .join(df_sla_coleta_entrega.select("id_pedido_ecommerce", "dias_coletado_entregue"), "id_pedido_ecommerce", "left") \
        .join(df_transportadoras_combinado, "id_pedido_ecommerce", "left")

    # --- APLICAÇÃO MASSIFICA DAS 10 REGRAS OFICIAIS ---
    df_silver_rastreio = (df_base
        .withColumn(
            "r1_id_rastreamento_falhou",
            # Nulo/vazio sempre reprova; duplicata só reprova a partir da
            # SEGUNDA ocorrência (qtd_id_rastreio, agora um row_number, > 1).
            # A primeira ocorrência é validada.
            F.col("id_rastreamento").isNull() | (F.col("id_rastreamento").cast("string") == "") | (F.col("qtd_id_rastreio") > 1)
        )
        .withColumn("r2_status_entrega_falhou", F.col("status_entrega").isNull() | (~F.col("status_entrega").isin(status_logísticos_validos)))
        .withColumn("r3_id_pedido_fk_falhou", F.col("id_pedido_ecommerce").isNull() | F.col("id_pedido_ref").isNull())
        .withColumn("r4_dt_evento_falhou", F.col("dt_evento_ts").isNull() | (F.col("dt_evento_ts") > F.current_timestamp()))
        .withColumn("r5_codigo_rastreio_falhou", F.col("codigo_rastreio").isNull() | (~F.col("codigo_rastreio").rlike(r"^[A-Z]{2}\d{9}$")))
        .withColumn("r6_ordem_cronologica_falhou", F.col("dt_evento_anterior").isNotNull() & (F.col("dt_evento_ts") < F.col("dt_evento_anterior")))
        .withColumn(
            "r7_consistencia_entrega_falhou",
            # CORREÇÃO: agora é uma checagem por PEDIDO (tem_evento_entregue),
            # não linha a linha contra "entregue".
            (F.col("status_pedido_ref") == "Entregue") & (F.coalesce(F.col("tem_evento_entregue"), F.lit(0)) == 0)
        )
        .withColumn(
            "r8_sla_violado_falhou",
            # CORREÇÃO: usa o intervalo específico entre 'coletado' e
            # 'entregue' do pedido, calculado à parte.
            (F.col("status_entrega") == "entregue") & (F.coalesce(F.col("dias_coletado_entregue"), F.lit(0)) > 30)
        )
        .withColumn("r9_pedido_cancelado_movimentado_falhou", (F.col("status_pedido_ref") == "Cancelado") & F.col("id_rastreamento").isNotNull())
        .withColumn(
            "r10_transportadora_inconsistente_falhou",
            # CORREÇÃO: usa a contagem combinada (histórico + lote), não só o lote atual.
            (F.col("status_pedido_ref") != "Cancelado") & (F.coalesce(F.col("qtd_transportadoras_total"), F.lit(1)) > 1)
        ))

    # MODIFICAÇÃO: Unificação total das 10 regras logísticas como Crítica
    regras_criticas = [
        "r1_id_rastreamento_falhou", "r2_status_entrega_falhou", "r3_id_pedido_fk_falhou",
        "r4_dt_evento_falhou", "r5_codigo_rastreio_falhou", "r6_ordem_cronologica_falhou",
        "r7_consistencia_entrega_falhou", "r8_sla_violado_falhou",
        "r9_pedido_cancelado_movimentado_falhou", "r10_transportadora_inconsistente_falhou"
    ]

    condicao_total_falha = reduce(lambda a, b: a | b, [F.col(c) for c in regras_criticas])

    df_silver_rastreio = (df_silver_rastreio
        .withColumn("silver_linha_valida", ~condicao_total_falha)
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(RUN_ID)))

    print("Muralha de qualidade aplicada com sucesso! "
          "R1: primeira ocorrência sobrevive. R6: ordem pelo fluxo de status. "
          "R7: checagem por pedido. R8: intervalo coletado->entregue específico. "
          "R10: contagem de transportadora com histórico acumulado.")
else:
    print("Nenhum dado logístico novo para aplicar regras.")

## Catálogo de Regras e Logs

In [0]:
if qtd_novos > 0:
    catalogo_regras = [
        {"coluna": "r1_id_rastreamento_falhou", "regra": "R1_ID_RASTREIO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_status_entrega_falhou", "regra": "R2_STATUS_ENTREGA_INVALIDO", "severidade": "Critica"},
        {"coluna": "r3_id_pedido_fk_falhou", "regra": "R3_ID_PEDIDO_FK_INEXISTENTE", "severidade": "Critica"},
        {"coluna": "r4_dt_evento_falhou", "regra": "R4_DATA_EVENTO_FUTURA_NULA", "severidade": "Critica"},
        {"coluna": "r5_codigo_rastreio_falhou", "regra": "R5_FORMATO_COD_RASTREIO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r6_ordem_cronologica_falhou", "regra": "R6_INVERSAO_CRONOLOGICA_STATUS", "severidade": "Critica"},
        {"coluna": "r7_consistencia_entrega_falhou", "regra": "R7_DIVERGENCIA_STATUS_ENTREGUE", "severidade": "Critica"},
        {"coluna": "r8_sla_violado_falhou", "regra": "R8_SLA_LOGISTICO_MAIOR_30_DIAS", "severidade": "Critica"},
        {"coluna": "r9_pedido_cancelado_movimentado_falhou", "regra": "R9_PEDIDO_CANCELADO_COM_EVENTO", "severidade": "Critica"},
        {"coluna": "r10_transportadora_inconsistente_falhou", "regra": "R10_MULTIPLAS_TRANSPORTADORAS_PEDIDO", "severidade": "Critica"}
    ]

    total_registros = df_silver_rastreio.count()
    logs_list = []

    for r in catalogo_regras:
        qtd_falhas = df_silver_rastreio.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, TABELA_ALVO, r["regra"], "FAIL", r["severidade"],
                int(qtd_falhas), int(total_registros), datetime.now(timezone.utc), f"Bronze Delta ({TABELA_ALVO})"
            ))

    # Schema dos logs estável
    # CORREÇÃO: havia um "\" sobrando depois de "qtd_registros_falhos", o que
    # deixava a string literal sem fechamento e quebrava o parser do Python.
    schema_final = StructType([
        StructField("run_id", StringType(), True),
        StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True),
        StructField("status", StringType(), True),
        StructField("severidade", StringType(), True),
        StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True),
        StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema=schema_final)
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_final)

    print("Logs de monitoramento consolidados e prontos para gravação.")
else:
    # CORREÇÃO: mesmo typo do "\" sobrando corrigido aqui também.
    schema_final_vazio = StructType([
        StructField("run_id", StringType(), True), StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True), StructField("status", StringType(), True),
        StructField("severidade", StringType(), True), StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True), StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_final_vazio)
    print("Micro-lote vazio: Nenhum log de monitoramento gerado.")

## Gravação Final via SDK

In [0]:
if qtd_novos > 0:
    # CORREÇÃO: "silver_tem_aviso" removido da lista para adequação ao novo esquema unificado
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]
    
    # ---------------- 1. GRAVAÇÃO DOS VÁLIDOS ---------------- #
    df_silver_validos = (df_silver_rastreio
        .filter(F.col("silver_linha_valida") == True)
        .select(*colunas_finais))
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=True
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada com sucesso!")

    # ---------------- 2. GRAVAÇÃO DA QUARENTENA (DEDUPLICADA) ---------------- #
    df_silver_invalidos = (df_silver_rastreio
        .filter(F.col("silver_linha_valida") == False)
        .select(*colunas_finais))
        
    if df_silver_invalidos.count() > 0:
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            
            # Garante o dedup correto pela chave primária natural de rastreamento
            df_quarentena_para_gravar = df_silver_invalidos.join(
                df_quarentena_historico.select("id_rastreamento"), 
                on="id_rastreamento", 
                how="left_anti"
            )
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            sucesso_quarentena = gravar_delta(
                df=df_quarentena_para_gravar,
                camada="silver/quarentena",
                tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS,
                mode="append",
                particionar=False 
            )
            if sucesso_quarentena:
                print(f"Enviados {qtd_novos_rejeitados} registros novos para a quarentena.")
        else:
            print("Todos os registros reprovados já existiam na quarentena histórica.")

    # ---------------- 3. GRAVAÇÃO DOS LOGS NA RAIZ ---------------- #
    if 'df_dq_monitoring_logs_novos' in locals() and df_dq_monitoring_logs_novos.count() > 0:
        sucesso_logs = gravar_delta(
            df=df_dq_monitoring_logs_novos, 
            camada="", 
            tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS, 
            mode="append", 
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade consolidados na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")

##  VALIDACAO


In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.orderBy(F.col("silver_processed_at").desc()).limit(20))
else:
    print(f"A tabela Silver {TABELA_ALVO} ainda não existe.")

if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
    df_logs_validacao = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS)
    df_logs_filtrados = df_logs_validacao.filter(F.col("tabela") == TABELA_ALVO)
    
    print(f"Logs na {TABELA_DQ} para {TABELA_ALVO}:", df_logs_filtrados.count())
    display(df_logs_filtrados.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print(f"Tabela {TABELA_DQ} ainda não existe no Data Lake.")